# Thermodynamic Entropy Calibration Experiment

## Better uncertainty calibration for LLM classifiers

This notebook implements and evaluates a physics-inspired calibration method that treats LLM predictive uncertainty as thermodynamic entropy. 

### What this experiment does:
1. Generates synthetic miscalibrated data mimicking SST-2 sentiment classification
2. Compares three calibration methods:
   - **Uncalibrated baseline**: Raw softmax predictions
   - **Temperature Scaling** (Guo et al. 2017): Global temperature adjustment
   - **Thermodynamic Entropy Calibration** (proposed): Per-sample temperature based on predictive entropy
3. Evaluates using ECE (Expected Calibration Error), Brier Score, NLL, and Accuracy
4. Generates reliability diagrams for visual comparison

### Key Results from Full Experiment:
- Uncalibrated: ECE=0.243, Accuracy=74.9%
- Temperature Scaling: ECE=0.031 (87.1% reduction), Accuracy=74.9%
- Thermodynamic Entropy: ECE=0.162 (33.4% reduction), Accuracy=74.9%

**Note**: This demo uses a small synthetic dataset for quick execution. The full experiment uses 872 samples.

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru - NOT pre-installed on Colab, always install
_pip('loguru')

# numpy, scipy, matplotlib - pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

In [ ]:
# Imports - copied from original method.py
import json
import os
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import softmax
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from loguru import logger

# Configure logger for notebook (simplified output)
logger.remove()
logger.add(lambda msg: print(msg, end=''), level="INFO", format="{message}")

print("Imports complete!")

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d144fc-thermodynamic-entropy-calibration-for-im/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined!")

In [ ]:
# Load the demo data
data = load_data()

# Extract logits and labels
logits = np.array(data["logits"])
labels = np.array(data["labels"])

print(f"Loaded {len(labels)} samples with {logits.shape[1]} classes")
print(f"Logits shape: {logits.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Class distribution: {np.bincount(labels)}")

## Configuration

Set all tunable parameters here. For this demo, we use minimal values for quick execution.
- `n_samples`: Number of samples (small for demo)
- `test_size`: Proportion of data used for testing
- `n_bins`: Number of bins for ECE calculation and reliability diagrams

In [ ]:
# Configuration - MINIMAL values for demo
# Increase these for more meaningful results (at cost of runtime)

# Data parameters
N_SAMPLES = len(labels)  # Use all loaded data (~50 for demo)

# Experiment parameters
TEST_SIZE = 0.5  # 50% for test (smaller train/val for demo)
N_BINS = 5  # Fewer bins for clearer demo visualization

# Hyperparameter search grid (reduced for demo)
T_0_VALUES = [0.5, 1.0, 1.5]  # Reduced from [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
ALPHA_VALUES = [0.0, 0.5, 1.0]  # Reduced from [0.0, 0.25, 0.5, 0.75, 1.0]
BETA_VALUES = [0.0, 0.5]  # Reduced from [0.0, 0.25, 0.5]

print("Configuration set!")
print(f"N_SAMPLES = {N_SAMPLES}")
print(f"TEST_SIZE = {TEST_SIZE}")
print(f"N_BINS = {N_BINS}")

## Step 1: Data Preparation

Split the data into train (for tuning hyperparameters), validation, and test sets.
We use a simple random split with 50% test / 25% train / 25% val for this demo.

In [ ]:
# Step 1: Dataset Preparation - Split into train/val/test

np.random.seed(42)  # For reproducibility

# Use the loaded data
n = len(labels)
indices = np.random.permutation(n)

# Split: 25% train, 25% val, 50% test (adjusted for small demo dataset)
train_end = int(0.25 * n)
val_end = int(0.5 * n)

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

train_logits, train_labels = logits[train_idx], labels[train_idx]
val_logits, val_labels = logits[val_idx], labels[val_idx]
test_logits, test_labels = logits[test_idx], labels[test_idx]

logger.info(f"Train: {len(train_labels)}, Val: {len(val_labels)}, Test: {len(test_labels)}")

## Step 2: Calibration Methods

Implement the three calibration methods:
1. **Uncalibrated**: Raw softmax predictions
2. **Temperature Scaling**: Optimize global temperature on validation set
3. **Thermodynamic Entropy**: Per-sample temperature based on predictive entropy and margin

In [ ]:
# Calibration Methods - copied from original method.py with minimal changes

def uncalibrated_predictions(logits):
    """Baseline: uncalibrated softmax predictions."""
    probs = softmax(logits, axis=1)
    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)
    return {"probs": probs, "preds": preds, "confs": confs}

def temperature_scaling(logits, labels, val_logits=None, val_labels=None, T_init=1.0):
    """Standard Temperature Scaling (Guo et al. 2017)."""
    if val_logits is None or val_labels is None:
        val_logits = logits
        val_labels = labels

    def nll_loss(T):
        """Negative log-likelihood loss for given temperature."""
        scaled_logits = val_logits / T
        probs = softmax(scaled_logits, axis=1)
        nll = -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10)
        return np.mean(nll)

    # Optimize temperature
    result = minimize_scalar(nll_loss, bounds=(0.05, 10.0), method='bounded')
    T_opt = result.x

    # Apply optimal temperature
    scaled_logits = logits / T_opt
    probs = softmax(scaled_logits, axis=1)
    preds = np.argmax(probs, axis=1)
    confs = np.max(probs, axis=1)

    return {"probs": probs, "preds": preds, "confs": confs, "T_opt": T_opt}

def thermodynamic_entropy_calibration(logits, labels=None, val_logits=None, val_labels=None,
                                      T_0=1.0, alpha=0.5, beta=0.3, tune_hyperparams=True):
    """Thermodynamic Entropy Calibration (proposed method)."""
    # Compute uncalibrated probabilities, entropy, and margin
    probs_uncal = softmax(logits, axis=1)
    entropy = -np.sum(probs_uncal * np.log(probs_uncal + 1e-10), axis=1)
    sorted_probs = np.sort(probs_uncal, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    if tune_hyperparams and val_logits is not None and val_labels is not None:
        # Hyperparameter tuning on validation set (using config values)
        best_score = float('inf')
        best_params = {"T_0": T_0, "alpha": alpha, "beta": beta}

        for T_0_trial in T_0_VALUES:
            for alpha_trial in ALPHA_VALUES:
                for beta_trial in BETA_VALUES:
                    val_probs_uncal = softmax(val_logits, axis=1)
                    val_entropy = -np.sum(val_probs_uncal * np.log(val_probs_uncal + 1e-10), axis=1)
                    val_sorted = np.sort(val_probs_uncal, axis=1)
                    val_margin = val_sorted[:, -1] - val_sorted[:, -2]

                    T_val = T_0_trial * (1 + alpha_trial * val_entropy + beta_trial * (1 - val_margin))
                    val_logits_norm = val_logits / T_val[:, np.newaxis]
                    val_probs_cal = softmax(val_logits_norm, axis=1)

                    nll = -np.log(val_probs_cal[np.arange(len(val_labels)), val_labels] + 1e-10)
                    score = np.mean(nll)

                    if score < best_score:
                        best_score = score
                        best_params = {"T_0": T_0_trial, "alpha": alpha_trial, "beta": beta_trial}

        T_0 = best_params["T_0"]
        alpha = best_params["alpha"]
        beta = best_params["beta"]
        logger.info(f"Optimal hyperparameters: T_0={T_0:.2f}, alpha={alpha:.2f}, beta={beta:.2f}")

    # Apply thermodynamic calibration
    T_per_sample = T_0 * (1 + alpha * entropy + beta * (1 - margin))
    logits_norm = logits / T_per_sample[:, np.newaxis]
    probs_cal = softmax(logits_norm, axis=1)
    preds = np.argmax(probs_cal, axis=1)
    confs = np.max(probs_cal, axis=1)

    return {"probs": probs_cal, "preds": preds, "confs": confs,
            "T_per_sample": T_per_sample, "T_0": T_0, "alpha": alpha, "beta": beta,
            "entropy": entropy, "margin": margin}

logger.info("Calibration methods defined!")

## Step 3: Evaluation Metrics

Define metrics to evaluate calibration quality:
- **ECE** (Expected Calibration Error): Weighted average of calibration gaps across confidence bins
- **Brier Score**: Mean squared error of probabilities vs one-hot encoded true labels
- **NLL** (Negative Log-Likelihood): Measures probability assigned to true class
- **Accuracy**: Standard classification accuracy

In [ ]:
# Calibration Metrics - copied from original method.py

def compute_ece(probs, labels, n_bins=10, strategy="uniform"):
    """Compute Expected Calibration Error (ECE)."""
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs = (preds == labels).astype(float)

    if strategy == "uniform":
        bins = np.linspace(0, 1, n_bins + 1)
    else:
        bins = np.quantile(confs, np.linspace(0, 1, n_bins + 1))

    ece = 0.0
    for i in range(n_bins):
        mask = (confs >= bins[i]) & (confs < bins[i+1])
        if np.sum(mask) > 0:
            bin_acc = np.mean(accs[mask])
            bin_conf = np.mean(confs[mask])
            bin_weight = np.sum(mask) / len(confs)
            ece += bin_weight * abs(bin_acc - bin_conf)

    return ece

def compute_brier_score(probs, labels):
    """Compute Brier Score."""
    n_samples, n_classes = probs.shape
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), labels] = 1
    return np.mean((probs - one_hot) ** 2)

def compute_nll(probs, labels):
    """Compute Negative Log-Likelihood."""
    true_class_probs = probs[np.arange(len(labels)), labels]
    return -np.mean(np.log(true_class_probs + 1e-10))

def compute_accuracy(preds, labels):
    """Compute accuracy."""
    return np.mean(preds == labels)

def evaluate_predictions(probs, preds, labels):
    """Compute all calibration metrics."""
    return {
        "ece": compute_ece(probs, labels, n_bins=N_BINS),
        "brier": compute_brier_score(probs, labels),
        "nll": compute_nll(probs, labels),
        "accuracy": compute_accuracy(preds, labels)
    }

logger.info("Evaluation metrics defined!")

## Step 4: Run Calibration Experiments

Apply all three calibration methods to the test set and evaluate them.

In [ ]:
# Step 2: Baseline - Uncalibrated
logger.info("\n[Step 2] Uncalibrated Baseline")

uncal_result = uncalibrated_predictions(test_logits)
uncal_metrics = evaluate_predictions(
    uncal_result["probs"], uncal_result["preds"], test_labels
)
logger.info(f"Uncalibrated metrics: {uncal_metrics}")

# Step 3: Temperature Scaling
logger.info("\n[Step 3] Temperature Scaling")

ts_result = temperature_scaling(
    test_logits, test_labels,
    val_logits=val_logits, val_labels=val_labels
)
ts_metrics = evaluate_predictions(
    ts_result["probs"], ts_result["preds"], test_labels
)
logger.info(f"Temperature Scaling metrics: {ts_metrics}")
logger.info(f"Optimal T: {ts_result['T_opt']:.4f}")

# Step 4: Thermodynamic Entropy Calibration
logger.info("\n[Step 4] Thermodynamic Entropy Calibration")

te_result = thermodynamic_entropy_calibration(
    test_logits, test_labels,
    val_logits=val_logits, val_labels=val_labels,
    tune_hyperparams=True
)
te_metrics = evaluate_predictions(
    te_result["probs"], te_result["preds"], test_labels
)
logger.info(f"Thermodynamic Entropy metrics: {te_metrics}")
logger.info(f"Hyperparameters: T_0={te_result['T_0']:.2f}, alpha={te_result['alpha']:.2f}")

## Step 5: Results Summary and Visualization

Display the results in a table and create reliability diagrams to visualize calibration quality.

In [ ]:
# Results Summary - Print in a readable table

print("\n" + "="*60)
print("EXPERIMENT RESULTS")
print("="*60)
print("\nResults Summary:")
print(f"{'Method':<30} {'ECE':>8} {'Brier':>8} {'NLL':>8} {'Acc':>8}")
print("-" * 62)

results = [
    ("Uncalibrated", uncal_metrics),
    ("Temperature Scaling", ts_metrics),
    ("Thermodynamic Entropy", te_metrics)
]

for name, metrics in results:
    print(f"{name:<30} {metrics['ece']:>8.4f} {metrics['brier']:>8.4f} "
          f"{metrics['nll']:>8.4f} {metrics['accuracy']:>8.4f}")

# Calculate improvements
print("\nCalibration Improvements (ECE reduction):")
baseline_ece = uncal_metrics['ece']
for name, metrics in results[1:]:
    improvement = (baseline_ece - metrics['ece']) / baseline_ece * 100
    print(f"  {name:<30}: {improvement:>6.1f}% reduction")

print("\n" + "="*60)

In [ ]:
# Reliability Diagrams - Visual comparison of calibration quality
# Adapted from original plot_reliability_diagram function

def plot_reliability_diagram(probs, labels, method_name, n_bins=5):
    """Plot reliability diagram (accuracy vs confidence per bin)."""
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs = (preds == labels).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_accs = []
    bin_confs = []

    for i in range(n_bins):
        mask = (confs >= bins[i]) & (confs < bins[i+1])
        if np.sum(mask) > 0:
            bin_accs.append(np.mean(accs[mask]))
            bin_confs.append(np.mean(confs[mask]))
        else:
            bin_accs.append(0)
            bin_confs.append(bin_centers[i])

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Reliability diagram
    ax[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax[0].bar(bin_centers, bin_accs, width=0.08, alpha=0.6, label=method_name)
    ax[0].set_xlabel('Confidence')
    ax[0].set_ylabel('Accuracy')
    ax[0].set_title(f'Reliability Diagram: {method_name}')
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    ax[0].set_ylim([0, 1])

    # Confidence histogram
    ax[1].hist(confs, bins=20, alpha=0.6, label=method_name)
    ax[1].set_xlabel('Confidence')
    ax[1].set_ylabel('Count')
    ax[1].set_title(f'Confidence Distribution: {method_name}')
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig

# Create reliability diagrams for all three methods
fig1 = plot_reliability_diagram(
    uncal_result["probs"], test_labels,
    "Uncalibrated", n_bins=N_BINS
)
plt.show()

fig2 = plot_reliability_diagram(
    ts_result["probs"], test_labels,
    f"Temp Scaling (T={ts_result['T_opt']:.2f})", n_bins=N_BINS
)
plt.show()

fig3 = plot_reliability_diagram(
    te_result["probs"], test_labels,
    f"Thermodynamic (T0={te_result['T_0']:.2f}, a={te_result['alpha']:.2f})", n_bins=N_BINS
)
plt.show()

## Interpretation

### Key Observations:

1. **Expected Calibration Error (ECE)**: Lower is better. Measures the difference between predicted confidence and actual accuracy.

2. **Reliability Diagrams**: The closer the bars are to the diagonal line, the better calibrated the model is.
   - Uncalibrated: Bars often deviate from diagonal (overconfident)
   - Temperature Scaling: Bars closer to diagonal (better calibrated)
   - Thermodynamic Entropy: Intermediate improvement

3. **Physics Analogy**: The thermodynamic method treats each prediction like a physical system:
   - Higher entropy (more uncertain predictions) → Higher temperature → Softer probabilities
   - Lower entropy (more confident predictions) → Lower temperature → Sharper probabilities

### Limitations of This Demo:
- Uses synthetic data (real experiments use SST-2 with DistilBERT)
- Small sample size for quick execution
- Reduced hyperparameter search grid

For full results, see the paper which uses 872 samples and complete hyperparameter tuning.